# Fine-Tuning LLMs for Toxicology Science
### A Complete Step-by-Step Tutorial

**Author:** Himanshu Goel | [himanshugoel.github.io](https://himanshugoel.github.io)

---

## Why fine-tune an LLM for toxicology?

General LLMs (GPT-4, Llama-3, Mistral) struggle with:

| Problem | Example | Impact |
|---------|---------|--------|
| Domain vocabulary | NOAEL, BMDL10, hERG IC50, TK6 MNT | Hallucinated values |
| Regulatory logic | ICH M7 two-method framework | Wrong safety calls |
| Quantitative precision | ALT 8-fold at 500 mg/kg | Missed numbers |
| Species extrapolation | Rat to human IVIVE | Incorrect guidance |

Fine-tuning adapts a pre-trained LLM using curated domain data,
achieving >90% task accuracy vs ~60-70% zero-shot on toxicology tasks.

## Fine-tuning strategies

```
FULL FINE-TUNING     QLoRA (4-bit + LoRA)    PROMPT TUNING
----------------     --------------------    ---------------
Update ALL weights   Small adapter matrices  Soft prompt tokens
7B = 28 GB VRAM      4-8 GB VRAM (T4/free)  < 1000 params
Best quality         Industry standard 2024  Limited quality
Needs A100           Works on Colab free     Legacy approach
```

**Recommendation:** QLoRA on Llama-3-8B or Mistral-7B.
Runs on T4 GPU (Colab free). Achieves >85% of full fine-tune quality.

## Steps covered

| Step | Content |
|------|---------|
| 1 | Setup and dependencies |
| 2 | Dataset design and curation (7 task types, 14 expert examples) |
| 3 | Data formatting: Alpaca / ChatML / Llama-3 |
| 4 | Data quality control pipeline |
| 5 | Base model selection guide |
| 6 | QLoRA: 4-bit quantisation + LoRA adapters |
| 7 | SFTTrainer: supervised fine-tuning loop |
| 8 | DPO: aligning with expert preferences |
| 9 | Evaluation: toxicology-specific benchmarks |
| 10 | Inference, deployment and safety guardrails |

---
## Section 2 -- Dataset Design for Toxicology Fine-Tuning

The **dataset is the most critical component**. A small, high-quality domain dataset (1,000-10,000 examples) outperforms a large noisy one.

### Task types

| Task | Format | Min examples | Priority |
|------|--------|-------------|----------|
| Question answering | Instruction-answer | 2000+ | HIGH |
| Structured extraction | Input text -> JSON | 1000+ | HIGH |
| NOAEL reasoning | Study data -> NOAEL + justification | 500+ | HIGH |
| ICH compliance | Scenario -> recommendation | 300+ | HIGH |
| IATA generation | Data package -> WoE narrative | 200+ | HIGH |
| Dose-response | Data -> expert explanation | 300+ | MEDIUM |
| Regulatory classification | Data -> GHS/CLP | 300+ | MEDIUM |

### Data sources (ranked by quality)

```
PRIMARY (curated, expert-reviewed)
  ECHA REACH Dossiers    NOAEL/LOAEL values, endpoint summaries
  NTP Technical Reports  Systematic study summaries
  EPA IRIS Assessments   Dose-response, cancer classifications
  FDA Drug Labels        Warnings, adverse effects
  ICH Guidelines         M7, S7, E14 -- regulatory logic Q&A

SECONDARY (needs cleaning)
  PubMed Abstracts       ToxCast, Tox21, journal papers
  Safety Data Sheets     GHS classification, first aid
  HSDB / TOXNET          Chemical hazard summaries
```

In [ ]:
# ── 2.1 Build a curated toxicology instruction dataset ──────────────────────
# 14 high-quality examples across 7 task types.
# In production: scale to 2000-10000 from ECHA/NTP/IRIS sources.

SYSTEM_PROMPT = (
    "You are ToxLLM, an expert AI assistant specialising in regulatory toxicology, "
    "pharmaceutical safety assessment, and chemical hazard evaluation. "
    "You provide accurate, evidence-based answers grounded in ICH, OECD, EPA, FDA, "
    "and ECHA guidelines."
)

TOX_DATASET = [

    # TASK 1: Q&A ──────────────────────────────────────────────────────────────
    {
        "task": "qa",
        "instruction": "You are an expert toxicologist. Answer accurately, citing relevant guidelines.",
        "input": "What is the ICH M7(R2) acceptable daily intake for a Class 1 nitrosamine impurity, and what is the scientific basis for this threshold?",
        "output": (
            "Per ICH M7(R2), Class 1 compounds (known human mutagens) have an acceptable daily intake "
            "(ADI) of 0.096 ng/day. This corresponds to a theoretical excess lifetime cancer risk of "
            "1 in 100,000 (10^-5) based on the Threshold of Toxicological Concern (TTC) concept, "
            "derived from the Carcinogenic Potency Database (CPDB) by identifying the most potent "
            "carcinogens and applying a 10^5 safety factor. For Class 2 compounds (known animal mutagens), "
            "the TTC is 1.5 ug/day unless a compound-specific acceptable intake can be justified "
            "using linear extrapolation from animal carcinogenicity data."
        ),
    },
    {
        "task": "qa",
        "instruction": "You are an expert toxicologist. Answer accurately, citing relevant guidelines.",
        "input": "Explain the CiPA initiative and how it replaces the traditional hERG-only cardiac safety paradigm.",
        "output": (
            "The Comprehensive In vitro Proarrhythmia Assay (CiPA) initiative, codified in ICH E14/S7B (2022), "
            "replaces single hERG IC50 testing with three tiers: Tier 1 uses in silico ion channel models "
            "integrating IC50 values for seven cardiac currents (IKr/hERG, INaL, ICaL, IKs, INaF, If, Ito) "
            "to compute a net pro-arrhythmic risk score. Tier 2 uses hiPS-CM on MEA, measuring DeltaFPDc "
            "(>20 ms = flag, >30 ms = high risk). Tier 3 provides in vivo confirmation only when needed. "
            "This reduces false positives from isolated hERG blockade by considering compensatory currents."
        ),
    },
    {
        "task": "qa",
        "instruction": "You are an expert toxicologist. Answer accurately, citing relevant guidelines.",
        "input": "What are the five OECD principles for regulatory QSAR acceptance?",
        "output": (
            "Per OECD GD 69 (2004, updated): (1) Defined endpoint -- unambiguous, measured under standard "
            "conditions. (2) Unambiguous algorithm -- fully described, reproducible. "
            "(3) Defined applicability domain (AD) -- compounds outside AD must be flagged; "
            "Tanimoto-based AD is most common. (4) Appropriate goodness-of-fit, robustness, and "
            "predictivity -- cross-validation Q2 >= 0.5 and external test set required; "
            "sensitivity >= 90% for ICH M7. (5) Mechanistic interpretation where possible."
        ),
    },
    {
        "task": "qa",
        "instruction": "You are an expert toxicologist. Answer accurately.",
        "input": "What is the IVIVE workflow used by EPA HTTK, and what parameters are required?",
        "output": (
            "EPA HTTK converts in vitro EC50 (uM) to in vivo administered equivalent doses. "
            "Required parameters: (1) fup -- fraction unbound in plasma by rapid equilibrium dialysis "
            "or predicted from LogP/MW; (2) CLint -- intrinsic hepatic clearance from HLM or hepatocytes; "
            "(3) Physiological constants: Qh=90 L/h, MPPGL=45 mg/g, liver weight=1500 g. "
            "Workflow: CLint -> CLh via well-stirred model [CLh = Qh*CLint*fup/(Qh+CLint*fup)]; "
            "AED = EC50_free * CLh / fup. The httk R package automates this for >1000 ToxCast chemicals."
        ),
    },

    # TASK 2: Structured Extraction ────────────────────────────────────────────
    {
        "task": "extraction",
        "instruction": "Extract structured toxicological data from the text and return valid JSON only.",
        "input": (
            "Chlorpyrifos by oral gavage to pregnant Sprague-Dawley rats at 1, 3, 10 mg/kg/day (GD6-GD20). "
            "Fetal brain cholinesterase inhibition: 35% at 1 mg/kg/day, 68% at 3 mg/kg/day, "
            "89% at 10 mg/kg/day. NOAEL=1 mg/kg/day; LOAEL=3 mg/kg/day."
        ),
        "output": (
            '{"compound": "chlorpyrifos", "species": "Sprague-Dawley rat", "sex": "pregnant female", '
            '"route": "oral gavage", "doses_mg_kg_day": [1, 3, 10], "exposure": "GD6-GD20", '
            '"critical_endpoint": "fetal brain cholinesterase inhibition", '
            '"che_inhibition": {"1": 35, "3": 68, "10": 89}, '
            '"noael_mg_kg_day": 1, "loael_mg_kg_day": 3}'
        ),
    },
    {
        "task": "extraction",
        "instruction": "Extract structured toxicological data from the text and return valid JSON only.",
        "input": (
            "Benzo[a]pyrene positive in Ames test (TA98, TA100 +S9): 8-fold and 12-fold increase at "
            "10 ug/plate. In vitro MNT (TK6 cells, +S9): positive at >= 0.3 uM (p<0.001), "
            "max MN frequency 4.8% vs 0.4% controls."
        ),
        "output": (
            '{"compound": "benzo[a]pyrene", "genotox": ['
            '{"assay": "Ames OECD TG 471", "s9": true, "result": "POSITIVE", '
            '"fold_increase": {"TA98": 8, "TA100": 12}, "dose": "10 ug/plate"}, '
            '{"assay": "in vitro MNT OECD TG 487", "s9": true, "result": "POSITIVE", '
            '"lowest_positive_uM": 0.3, "max_mn_pct": 4.8, "control_pct": 0.4}], '
            '"ich_m7_class": "Class 2", "overall": "POSITIVE"}'
        ),
    },

    # TASK 3: NOAEL Reasoning ──────────────────────────────────────────────────
    {
        "task": "noael_reasoning",
        "instruction": "Identify the NOAEL and LOAEL with scientific justification per OECD guidance.",
        "input": (
            "Compound X, oral, F344 rats, 28 days, 0/10/50/200 mg/kg/day (10/sex/group). "
            "200 mg/kg/day: body weight -18% (p<0.001), liver weight +32% (p<0.001), "
            "centrilobular hepatocyte hypertrophy 9/10, ALT 3.2x ULN. "
            "50 mg/kg/day: liver weight +12% (p<0.05), minimal hypertrophy 3/10, ALT 1.1x ULN. "
            "10 mg/kg/day: no findings."
        ),
        "output": (
            "NOAEL: 10 mg/kg/day. LOAEL: 50 mg/kg/day.\n\n"
            "Justification: NOAEL=10 mg/kg/day -- no treatment-related findings.\n\n"
            "LOAEL=50 mg/kg/day based on: (1) liver weight +12% (p<0.05), exceeding the "
            "10% adaptive/adverse threshold; (2) minimal hepatocyte hypertrophy in 3/10 animals. "
            "ALT not significantly elevated, but morphological findings alone suffice for LOAEL.\n\n"
            "200 mg/kg/day: frank toxicity (body weight -18%, severe liver pathology, ALT >3x ULN). "
            "Applying 100-fold uncertainty factor (x10 inter-species, x10 intra-human) to NOAEL "
            "yields guidance value of 0.1 mg/kg/day."
        ),
    },

    # TASK 4: ICH Compliance ───────────────────────────────────────────────────
    {
        "task": "ich_compliance",
        "instruction": "Apply the relevant ICH guideline and provide a clear regulatory recommendation.",
        "input": (
            "A new drug has a secondary N-nitroso impurity at 15 ppm in drug substance. "
            "Intended dose: 50 mg/day. Is this acceptable under ICH M7(R2)?"
        ),
        "output": (
            "Daily intake = 50 mg/day x 15 ppm = 750 ng/day.\n\n"
            "ICH M7(R2): N-nitroso moiety on secondary amine = Class 1 structural alert "
            "(known human mutagen). Acceptable intake Class 1 = 0.096 ng/day.\n\n"
            "Exceedance: 750 / 0.096 = 7,813-fold. NOT acceptable.\n\n"
            "Required specification: 0.096 ng/day / 50 mg/day = 1.9 ppb.\n\n"
            "Actions: (1) reduce to <=1.9 ppb; (2) root cause: eliminate nitrosation pathway; "
            "(3) validate LC-MS/MS with LOQ <=0.3 ppb per FDA 2021 guidance."
        ),
    },
    {
        "task": "ich_compliance",
        "instruction": "Apply the relevant ICH guideline and provide a clear regulatory recommendation.",
        "input": (
            "Phase III drug: QTc prolongation 18 ms mean, upper 90% CI = 22 ms in TQT study. "
            "ICH E14 assessment and next steps?"
        ),
        "output": (
            "ICH E14 Assessment: Mean QTc 18 ms (>10 ms = positive TQT). "
            "Upper CI 22 ms (>20 ms = substantial concern).\n\n"
            "Conclusion: Positive TQT with substantial QTc concern.\n\n"
            "Actions:\n"
            "1. Labelling: QTc prolongation warning in Warnings/Precautions (18 ms mean).\n"
            "2. Contraindications: concomitant QT-prolonging drugs.\n"
            "3. Exposure-response modelling: concentration-QTc relationship.\n"
            "4. CiPA Tier 2: hiPS-CM MEA to confirm mechanism.\n"
            "5. For oncology: development may continue with ECG monitoring per ICH E14 (2023 update)."
        ),
    },

    # TASK 5: IATA Generation ──────────────────────────────────────────────────
    {
        "task": "iata_generation",
        "instruction": "Generate an IATA weight-of-evidence assessment narrative per OECD GD 255.",
        "input": (
            "Compound Z: (1) ICH M7 structural alerts: none; (2) QSAR Ames: P=0.08, within AD; "
            "(3) DPRA depletion: 3.2% (negative, threshold 6.38%); (4) KeratinoSens: Imax 89% "
            "(negative, threshold 150%); (5) h-CLAT: negative; (6) liver organoid 14d 10uM: "
            "ATP 94%, LDH 8%, albumin 18.2 ug/mL/d; (7) hERG QSAR: IC50 >30 uM."
        ),
        "output": (
            "IATA WoE Assessment -- Compound Z\n\n"
            "Genotoxicity (ICH M7(R2) two-method):\n"
            "Method 1 (rule-based): No structural alerts. Provisional Class 5.\n"
            "Method 2 (QSAR): Ames P=0.08 (negative, within AD).\n"
            "Conclusion: Both concordantly negative. ICH M7 Class 5. No further testing required.\n\n"
            "Skin Sensitisation (OECD TG 497 DA2 -- 2o3):\n"
            "DPRA: Negative (3.2% < 6.38%). KeratinoSens: Negative (89% < 150%). h-CLAT: Negative.\n"
            "Conclusion: 0/3 positive. NON-SENSITISER. No LLNA required.\n\n"
            "Hepatotoxicity (liver organoid, 14d, 10 uM):\n"
            "ATP 94% (>80%), LDH 8% (<20%), albumin 18.2 (>12 threshold). DILI risk: LOW.\n\n"
            "Cardiac (CiPA Tier 1): hERG IC50 >30 uM. Occupancy <3% at free Cmax. No Tier 2 needed.\n\n"
            "Overall: NO CONCERN for genotoxicity, skin sensitisation, hepatotoxicity, or cardiac. "
            "No animal studies required. Compound may progress to Phase I."
        ),
    },

    # TASK 6: Dose-Response ────────────────────────────────────────────────────
    {
        "task": "dose_response",
        "instruction": "Interpret the dose-response data and explain the regulatory significance.",
        "input": (
            "hERG IC50 assay for compound ABC: concentrations (uM): 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0. "
            "Inhibition (%): 2, 4, 9, 22, 48, 71, 88, 96. Hill n=1.3. IC50=0.95 uM. Free Cmax=0.1 uM."
        ),
        "output": (
            "Curve: sigmoidal Hill fit, IC50=0.95 uM, Hill n=1.3 (slight positive cooperativity).\n\n"
            "Safety margin: IC50/Cmax,free = 0.95/0.1 = 9.5-fold.\n\n"
            "ICH S7B: margin <10-fold triggers cardiac safety flag. At 9.5-fold, FAILS criterion.\n\n"
            "At Cmax (0.1 uM): occupancy = 1/(1+9.5) = ~9.5% -- borderline low-concern threshold.\n\n"
            "CiPA: Proceed with Tier 1 multi-channel IC50 panel. ICaL blockade (common for CaCC) "
            "would offset IKr risk. Flag for ECG monitoring in Phase I TQT study per ICH E14."
        ),
    },

    # TASK 7: Regulatory Classification ───────────────────────────────────────
    {
        "task": "classification",
        "instruction": "Provide EU CLP/GHS hazard classification with justification.",
        "input": (
            "Compound data: LD50 oral rat=180 mg/kg; inhalation LC50 rat 4h=0.8 mg/L vapour; "
            "skin sensitisation LLNA EC3=0.35%; Ames positive; "
            "reproductive LOAEL=200 mg/kg/day (decreased sperm motility); "
            "Daphnia 48h LC50=0.8 mg/L; not readily biodegradable."
        ),
        "output": (
            "Acute Tox Oral: LD50=180 mg/kg -> Category 3 (25-300). H301 Toxic if swallowed.\n"
            "Acute Tox Inhalation: LC50=0.8 mg/L/4h vapour -> Category 2 (0.5-2.0). H330 Fatal if inhaled.\n"
            "Skin Sensitisation: LLNA EC3=0.35% (0.1-1.0%) -> Category 1B. H317.\n"
            "Germ Cell Mutagenicity: Ames positive, no in vivo data -> Category 2. H341.\n"
            "Reproductive Toxicity: decreased sperm motility -> Category 2. H361.\n"
            "Aquatic Acute: Daphnia 0.8 mg/L (<1 mg/L) -> Category 1. H400.\n"
            "Aquatic Chronic: not biodegradable + LC50 <1 mg/L -> Chronic Cat 1. H410.\n"
            "Summary: H301, H317, H330, H341, H361, H400, H410. Signal word: DANGER."
        ),
    },
]

print(f"Dataset: {len(TOX_DATASET)} examples")
from collections import Counter
print(Counter(d['task'] for d in TOX_DATASET))

---
## Section 3 -- Data Formatting

Different models require different prompt formats. Wrong template = 15-30% performance loss.

```
ALPACA format (Llama, Vicuna)
  ### Instruction: {instruction}
  ### Input: {input}
  ### Response: {output}

CHATML format (Mistral, Qwen, OpenHermes)
  <|im_start|>system
  {system}<|im_end|>
  <|im_start|>user
  {user}<|im_end|>
  <|im_start|>assistant
  {output}<|im_end|>

LLAMA-3 format (Meta Llama 3)
  <|begin_of_text|><|start_header_id|>system<|end_header_id|>
  {system}<|eot_id|><|start_header_id|>user<|end_header_id|>
  {user}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
  {response}<|eot_id|>
```

In [ ]:
SYSTEM_PROMPT_TOX = (
    'You are ToxLLM, an expert AI assistant specialising in regulatory toxicology, '
    'pharmaceutical safety assessment, and chemical hazard evaluation. '
    'You provide accurate, evidence-based answers grounded in ICH, OECD, EPA, FDA, '
    'and ECHA guidelines.'
)

def format_alpaca(ex):
    if ex.get('input','').strip():
        return (f'### Instruction:\n{ex["instruction"]}\n\n'
                f'### Input:\n{ex["input"]}\n\n'
                f'### Response:\n{ex["output"]}')
    return f'### Instruction:\n{ex["instruction"]}\n\n### Response:\n{ex["output"]}'

def format_chatml(ex, system=SYSTEM_PROMPT_TOX):
    user = ex['instruction']
    if ex.get('input','').strip():
        user += f'\n\n{ex["input"]}'
    return (f'<|im_start|>system\n{system}<|im_end|>\n'
            f'<|im_start|>user\n{user}<|im_end|>\n'
            f'<|im_start|>assistant\n{ex["output"]}<|im_end|>')

def format_llama3(ex, system=SYSTEM_PROMPT_TOX):
    user = ex['instruction']
    if ex.get('input','').strip():
        user += f'\n\n{ex["input"]}'
    return (f'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n'
            f'{system}<|eot_id|>'
            f'<|start_header_id|>user<|end_header_id|>\n'
            f'{user}<|eot_id|>'
            f'<|start_header_id|>assistant<|end_header_id|>\n'
            f'{ex["output"]}<|eot_id|>')

# Show all three on first example
ex = TOX_DATASET[0]
print('ALPACA:'); print(format_alpaca(ex)[:300]); print('...')
print('\nCHATML:'); print(format_chatml(ex)[:300]); print('...')
print('\nLLAMA3:'); print(format_llama3(ex)[:300]); print('...')

In [ ]:
import random
def build_dataset(data, fmt='chatml', train=0.85, val=0.10, seed=42):
    random.seed(seed)
    shuffled = data.copy(); random.shuffle(shuffled)
    fmtfn = {'alpaca':format_alpaca,'chatml':format_chatml,'llama3':format_llama3}.get(fmt,format_chatml)
    formatted = [{'text':fmtfn(ex),'task':ex.get('task','general'),
                  'n_tokens':int(len(fmtfn(ex).split())*1.3)} for ex in shuffled]
    n = len(formatted); n_tr = int(n*train); n_v = int(n*val)
    splits = {'train':formatted[:n_tr],'val':formatted[n_tr:n_tr+n_v],'test':formatted[n_tr+n_v:]}
    for sn, sd in splits.items():
        avg = sum(d['n_tokens'] for d in sd)/max(1,len(sd))
        print(f'  {sn:6s}: {len(sd):3d} examples | avg ~{avg:.0f} tokens')
    return splits

import numpy as np
print('Building dataset (chatml):')
splits = build_dataset(TOX_DATASET)

# Save to JSONL
for name, data in splits.items():
    path = f'tox_finetune_{name}.jsonl'
    with open(path,'w') as f:
        for ex in data: f.write(json.dumps(ex)+'\n')
    print(f'Saved {path}')

# Plot token distribution
all_toks = [d['n_tokens'] for s in splits.values() for d in s]
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,4))
ax.hist(all_toks, bins=15, color='#1565C0', alpha=0.8, edgecolor='white')
ax.axvline(2048, color='#E74C3C', lw=2, linestyle='--', label='2048 token limit')
ax.set_xlabel('Approx tokens'); ax.set_ylabel('Examples')
ax.set_title('Training Data Token Length Distribution', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## Section 4 -- Data Quality Control

Garbage in, garbage out. Run every example through QC before training.

In [ ]:
import re, json, numpy as np

class ToxDataQC:
    MIN_OUT = 30; MAX_OUT = 2048
    HALLUCINATION = [
        r"I (?:don't|do not) (?:know|have access)",
        r'As of my (?:knowledge|training)',
        r"I (?:cannot|can't) (?:provide|give)",
    ]
    QUALITY_MARKERS = [
        r'\b(?:mg/kg|uM|nM|ppm|ppb|ng/day|ug/day)\b',
        r'\b(?:NOAEL|LOAEL|BMDL|IC50|EC50|LD50|LC50)\b',
        r'\b(?:ICH|OECD|EPA|FDA|ECHA|GHS|CLP)\b',
        r'\b(?:p\s*<|statistically significant)\b',
    ]

    def check(self, ex, idx=0):
        flags = []; score = 1.0
        out = ex.get('output',''); instr = ex.get('instruction','')
        n_out = len(out.split())
        if n_out < self.MIN_OUT: flags.append(f'short_output:{n_out}'); score -= 0.3
        if n_out > self.MAX_OUT: flags.append('long_output'); score -= 0.15
        if not instr: flags.append('missing_instruction'); score -= 0.4
        for p in self.HALLUCINATION:
            if re.search(p, out, re.IGNORECASE):
                flags.append('hallucination'); score -= 0.3; break
        q_hits = sum(1 for p in self.QUALITY_MARKERS
                     if re.search(p, out + ex.get('input',''), re.IGNORECASE))
        if q_hits == 0 and n_out > 50: flags.append('no_quant_markers'); score -= 0.1
        if not re.search(r'[.!?"\']\s*$', out.strip()):
            flags.append('possibly_truncated'); score -= 0.05
        if ex.get('task') == 'extraction':
            try: json.loads(out.strip())
            except: flags.append('invalid_json'); score -= 0.3
        score = round(max(0, min(1, score)), 3)
        return {'passed': score >= 0.6 and 'missing_instruction' not in ' '.join(flags),
                'score': score, 'flags': flags}

    def run(self, data):
        results = [self.check(ex, i) for i, ex in enumerate(data)]
        passed  = [data[i] for i, r in enumerate(results) if r['passed']]
        scores  = [r['score'] for r in results]
        flags   = {}
        for r in results:
            for f in r['flags']: k = f.split(':')[0]; flags[k] = flags.get(k,0)+1
        print(f'QC: {len(data)} -> {len(passed)} passed ({len(passed)/len(data)*100:.0f}%)')
        print(f'Mean score: {np.mean(scores):.3f} | Flags: {flags}')
        return passed

def dedup(data, threshold=0.85):
    seen=[]; unique=[]
    for ex in data:
        words = ex.get('output','').lower().split()
        ngrams = set(tuple(words[i:i+4]) for i in range(len(words)-3))
        if not any(len(ngrams & s)/max(1,len(ngrams|s))>=threshold for s in seen):
            seen.append(ngrams); unique.append(ex)
    print(f'Dedup: kept {len(unique)}/{len(data)}')
    return unique

qc = ToxDataQC()
clean = qc.run(TOX_DATASET)
clean = dedup(clean)
print(f'Final: {len(clean)} examples')

---
## Section 5 -- Base Model Selection

| Model | Size | VRAM (4-bit) | Biomedical | Licence | Best for |
|-------|------|-------------|-----------|---------|----------|
| Llama-3-8B-Instruct | 8B | 6 GB | Good | Llama 3 Community | General regulatory |
| **Mistral-7B-Instruct-v0.3** | 7B | 5 GB | Good | **Apache 2.0** | Production (no gating) |
| BioMistral-7B | 7B | 5 GB | Excellent | Apache 2.0 | Clinical/biomedical |
| Llama-3-70B-Instruct | 70B | 40 GB | Very Good | Llama 3 | Highest quality |

**Recommendation:** Start with `mistralai/Mistral-7B-Instruct-v0.3` -- Apache 2.0, no access gating.

In [ ]:
SELECTED_MODEL = {
    'model_id':    'mistralai/Mistral-7B-Instruct-v0.3',
    'format':      'chatml',
    'context_len': 32768,
    'vram_4bit_gb': 5,
}
print(f'Selected: {SELECTED_MODEL["model_id"]}')
print(f'Format: {SELECTED_MODEL["format"]} | VRAM (4-bit): {SELECTED_MODEL["vram_4bit_gb"]} GB')

if HF_AVAILABLE and TORCH_AVAILABLE:
    from transformers import BitsAndBytesConfig
    # 4-bit NF4 quantisation: reduces ~28 GB (bf16) to ~5 GB
    bnb_config = BitsAndBytesConfig(
        load_in_4bit              = True,
        bnb_4bit_quant_type       = 'nf4',
        bnb_4bit_compute_dtype    = torch.bfloat16,  # use float16 for T4
        bnb_4bit_use_double_quant = True,
    )
    tokenizer = AutoTokenizer.from_pretrained(
        SELECTED_MODEL['model_id'], trust_remote_code=True, padding_side='right')
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print(f'Vocab: {tokenizer.vocab_size:,} | Pad: {tokenizer.pad_token}')
    ex_toks = tokenizer.encode(format_chatml(TOX_DATASET[0]))
    print(f'Example 1: {len(ex_toks)} tokens')
else:
    print('Load template: AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config)')

# Training cost estimate
total_chars = sum(len(format_chatml(ex)) for ex in TOX_DATASET)
total_toks  = int(total_chars / 3.5)
print(f'Total tokens: {total_toks:,}')
print(f'T4 estimate: ~{total_toks*3/800_000:.0f} min (3 epochs)')

---
## Section 6 -- QLoRA Configuration

LoRA adds small trainable matrices to frozen model weights:

```
W' = W + B @ A   where B in R(d x r), A in R(r x d), rank r << d
Mistral-7B: 7B params x 4 bits = 3.5 GB frozen
LoRA adapters (r=32): ~150 MB trainable  (<1% of model)
```

### LoRA hyperparameter guide

| Param | Value | Why |
|-------|-------|-----|
| r | 32-64 | Higher = more capacity. 32 general, 64 for large domain shift |
| lora_alpha | 2*r | Effective scale = alpha/r |
| target_modules | q,k,v,o,gate,up,down | All attention + FFN layers |
| lora_dropout | 0.05 | Light regularisation |
| bias | 'none' | Not needed for SFT |

In [ ]:
if HF_AVAILABLE:
    from peft import LoraConfig, TaskType
    lora_config = LoraConfig(
        r              = 32,
        lora_alpha     = 64,
        target_modules = ['q_proj','k_proj','v_proj','o_proj',
                          'gate_proj','up_proj','down_proj'],
        lora_dropout   = 0.05,
        bias           = 'none',
        task_type      = TaskType.CAUSAL_LM,
    )
    print(f'Rank: {lora_config.r} | Alpha: {lora_config.lora_alpha} | Scale: {lora_config.lora_alpha/lora_config.r:.1f}')
    print(f'Modules: {lora_config.target_modules}')
else:
    print('peft not installed: pip install peft')

# Parameter efficiency
d_model=4096; n_layers=32; n_mods=7; r=32
lora_params = n_layers * n_mods * 2 * d_model * r
total = 7_000_000_000
print(f'Total params: {total/1e9:.1f}B | LoRA: {lora_params/1e6:.0f}M ({lora_params/total*100:.2f}%)')
print(f'Memory saving: {(1-lora_params/total)*100:.0f}% vs full fine-tuning')

---
## Section 7 -- SFTTrainer: Supervised Fine-Tuning

`SFTTrainer` handles training loop, gradient accumulation, mixed precision, checkpointing.

**Critical:** `DataCollatorForCompletionOnlyLM` masks prompt tokens -- the model **only learns to generate responses**, not reproduce instructions. Without this, training efficiency drops ~50%.

In [ ]:
if HF_AVAILABLE and TORCH_AVAILABLE:
    from transformers import TrainingArguments
    training_args = TrainingArguments(
        output_dir                  = './toxllm_output',
        num_train_epochs            = 3,
        per_device_train_batch_size = 2,       # 2 on T4; 4 on A100
        gradient_accumulation_steps = 8,       # effective batch = 2x8 = 16
        learning_rate               = 2e-4,    # QLoRA standard
        warmup_ratio                = 0.05,
        lr_scheduler_type           = 'cosine',
        optim                       = 'paged_adamw_32bit',
        max_grad_norm               = 0.3,     # gradient clipping
        weight_decay                = 0.001,
        bf16                        = (DEVICE == 'cuda'),
        eval_strategy               = 'steps',
        eval_steps                  = 50,
        save_steps                  = 50,
        save_total_limit            = 3,
        load_best_model_at_end      = True,
        logging_steps               = 10,
        report_to                   = 'none',
        seed                        = 42,
    )
    print(f'Effective batch: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}')
    print(f'LR: {training_args.learning_rate} | Scheduler: {training_args.lr_scheduler_type}')

print('''
# ── SFTTrainer setup ────────────────────────────────────────────────────────
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

RESPONSE_TEMPLATE = '<|im_start|>assistant'  # ChatML
# Llama-3: '<|start_header_id|>assistant<|end_header_id|>'
# Alpaca:  '### Response:'

collator = DataCollatorForCompletionOnlyLM(
    response_template = RESPONSE_TEMPLATE,
    tokenizer         = tokenizer,
)

trainer = SFTTrainer(
    model              = model,          # 4-bit QLoRA model
    tokenizer          = tokenizer,
    args               = training_args,
    train_dataset      = train_hf,       # HF Dataset with text field
    eval_dataset       = val_hf,
    dataset_text_field = 'text',
    max_seq_length     = 2048,
    data_collator      = collator,
)

trainer.train()
trainer.save_model('./toxllm_lora_adapter')     # save adapter only (~150 MB)
merged = model.merge_and_unload()               # merge for deployment
merged.save_pretrained('./toxllm_merged')
''')

# Visualise expected training curve
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
steps = np.arange(0, 300, 10)
tl = np.clip(2.8*np.exp(-steps/100)+0.45+np.random.normal(0,0.05,len(steps)),0.4,3)
vl = np.clip(2.9*np.exp(-steps/110)+0.55+np.random.normal(0,0.04,len(steps)),0.5,3)
lr = 2e-4 * (np.cos(np.pi*steps/steps.max())+1)/2
fig, axes = plt.subplots(1,2,figsize=(13,4))
axes[0].plot(steps,tl,'#1565C0',lw=2.2,label='Train loss')
axes[0].plot(steps,vl,'#E74C3C',lw=2.2,label='Val loss')
axes[0].set_xlabel('Steps'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training Curve (simulated, 3 epochs)', fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(steps,lr*1e4,'#8E44AD',lw=2.2)
axes[1].set_xlabel('Steps'); axes[1].set_ylabel('LR (x1e-4)')
axes[1].set_title('Cosine LR with 5% Warmup', fontweight='bold')
axes[1].grid(True, alpha=0.3)
plt.suptitle('Expected QLoRA Training Dynamics', fontweight='bold')
plt.tight_layout(); plt.show()
print('Expected final val loss: 0.55-0.65 for 7B model on toxicology data')

---
## Section 8 -- DPO: Aligning with Expert Preferences

After SFT the model may give vague or poorly structured answers. DPO teaches the model to prefer expert-quality responses using comparison pairs -- no reward model needed.

```
Each DPO pair:
  prompt    = the question
  chosen    = specific, quantitative, regulatory-grounded  (expert)
  rejected  = vague, missing numbers, no guideline refs    (weak)
```

In [ ]:
DPO_DATA = [
    {
        'prompt': '<|im_start|>user\nWhat is the NOAEL significance in risk assessment?<|im_end|>\n<|im_start|>assistant\n',
        'chosen': (
            'The NOAEL is the highest tested dose producing no statistically significant adverse effect. '
            'Guidance values are derived by dividing NOAEL by composite uncertainty factors '
            '(typically 100 = x10 inter-species x10 intra-human, per IPCS/WHO). '
            'Limitation: NOAEL is constrained to tested doses and ignores curve shape. '
            'The BMD approach (EFSA mandate) is preferred as it uses all dose-response data; '
            'BMDL (lower 95% CI) is used as the point of departure.'
        ),
        'rejected': 'The NOAEL is an important concept that helps determine safe dose levels. Regulatory agencies use it to assess risk.',
    },
    {
        'prompt': '<|im_start|>user\nCompound A: hERG IC50=2uM, free Cmax=0.5uM. Cardiac concern?<|im_end|>\n<|im_start|>assistant\n',
        'chosen': (
            'Yes -- cardiac safety concern. IC50:Cmax,free = 2/0.5 = 4-fold, below the 10-fold ICH S7B threshold. '
            'Channel occupancy at Cmax = 1/(1+IC50/Cmax) = 1/(1+4) = 20%. '
            'Required: CiPA Tier 1 (7-channel IC50 panel), then Tier 2 (hiPS-CM MEA if DeltaFPDc >20 ms). '
            'Flag for Phase I ECG monitoring per ICH E14.'
        ),
        'rejected': 'The hERG IC50 of 2uM with Cmax of 0.5uM could potentially be concerning. Further cardiac studies are recommended.',
    },
    {
        'prompt': '<|im_start|>user\nN-nitroso impurity 15 ppm in 50 mg/day drug. ICH M7?<|im_end|>\n<|im_start|>assistant\n',
        'chosen': (
            'Daily intake = 50 mg x 15e-6 = 750 ng/day. ICH M7(R2): N-nitroso = Class 1 (known human mutagen). '
            'Acceptable = 0.096 ng/day. Exceedance: 750/0.096 = 7,813-fold. NOT acceptable. '
            'Required spec: <=0.096/50 mg/day = 1.9 ppb. '
            'Actions: (1) reduce spec; (2) eliminate nitrosation pathway; (3) validate LC-MS/MS, LOQ <=0.3 ppb.'
        ),
        'rejected': 'The N-nitroso alert is a Class 1 concern per ICH M7. A thorough risk assessment is recommended to evaluate the impurity level.',
    },
]

print(f'DPO pairs: {len(DPO_DATA)}')
for i, p in enumerate(DPO_DATA):
    cw = len(p['chosen'].split()); rw = len(p['rejected'].split())
    print(f'  Pair {i+1}: chosen={cw} words | rejected={rw} | ratio={cw/rw:.1f}x')

print('''
# DPO Training
from trl import DPOTrainer, DPOConfig

dpo_config = DPOConfig(
    beta=0.1,             # lower = stronger preference signal
    max_length=2048,
    max_prompt_length=512,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,   # 10-40x lower than SFT
    num_train_epochs=1,
    output_dir='./toxllm_dpo',
)

dpo_trainer = DPOTrainer(
    model=sft_model,      # SFT-trained model
    ref_model=None,       # None = frozen SFT as reference
    args=dpo_config,
    train_dataset=dpo_dataset,
    tokenizer=tokenizer,
)
dpo_trainer.train()
''')

---
## Section 9 -- Evaluation: Toxicology-Specific Metrics

Standard BLEU/ROUGE miss domain quality. Three additional metrics matter for toxicology:

| Metric | What it measures | Why critical |
|--------|-----------------|-------------|
| ROUGE-L | Lexical overlap | Baseline text quality |
| Quantitative recall | Are doses/IC50s/fold-changes preserved? | Regulatory accuracy |
| Regulatory F1 | Correct ICH/OECD citations? | Compliance validity |
| Composite | 0.30*ROUGE + 0.25*Quant + 0.25*Reg + 0.20*Field | Overall quality |

Expected benchmark improvement:

| Stage | ROUGE-L | Quant recall |
|-------|---------|-------------|
| Zero-shot | ~0.28 | ~0.41 |
| After SFT | ~0.62 | ~0.78 |
| After DPO | ~0.68 | ~0.82 |

In [ ]:
import re, json, numpy as np

def evaluate_tox_response(prediction, reference, task='qa'):
    scores = {}
    # ROUGE-L (lexical overlap)
    try:
        from rouge_score import rouge_scorer
        sc = rouge_scorer.RougeScorer(['rougeL','rouge1'], use_stemmer=True)
        r  = sc.score(reference, prediction)
        scores['rouge_L'] = round(r['rougeL'].fmeasure, 4)
        scores['rouge_1'] = round(r['rouge1'].fmeasure, 4)
    except ImportError:
        ref_t = set(reference.lower().split())
        pre_t = set(prediction.lower().split())
        scores['rouge_L'] = round(len(ref_t & pre_t)/max(1,max(len(ref_t),len(pre_t))), 4)
        scores['rouge_1'] = round(len(ref_t & pre_t)/max(1,len(ref_t)), 4)

    # Quantitative value recall
    num_pat  = r'\d+(?:\.\d+)?\s*(?:mg/kg|uM|nM|fold|%|ng/day|ug/day|ppb|ppm)?'
    ref_nums = set(re.findall(num_pat, reference, re.IGNORECASE))
    pre_nums = set(re.findall(num_pat, prediction, re.IGNORECASE))
    scores['quant_recall'] = round(len(ref_nums & pre_nums)/max(1,len(ref_nums)), 4)

    # Regulatory citation F1
    reg_pat  = r'ICH\s+[SMQE]\d+\w*|OECD\s+(?:TG|GD)\s*\d+|EPA|CLP|GHS|EFSA'
    ref_regs = set(re.findall(reg_pat, reference, re.IGNORECASE))
    pre_regs = set(re.findall(reg_pat, prediction, re.IGNORECASE))
    if ref_regs:
        prec = len(ref_regs & pre_regs)/max(1,len(pre_regs))
        rec  = len(ref_regs & pre_regs)/len(ref_regs)
        scores['reg_f1'] = round(2*prec*rec/max(1e-9,prec+rec), 4)
    else: scores['reg_f1'] = 1.0

    # JSON field coverage (extraction tasks)
    if task == 'extraction':
        try:
            rj = json.loads(reference); pj = json.loads(prediction)
            scores['field_cov'] = round(len(set(rj)&set(pj))/max(1,len(rj)), 4)
        except: scores['field_cov'] = 0.0
    else: scores['field_cov'] = scores['rouge_L']

    # Composite score
    scores['composite'] = round(
        0.30*scores['rouge_L'] + 0.25*scores['quant_recall'] +
        0.25*scores['reg_f1'] + 0.20*scores['field_cov'], 4)
    return scores

def sim_pred(ex, quality=0.75):
    ref = ex['output']; words = ref.split()
    kept = ' '.join(words[:int(len(words)*quality)])
    if quality < 0.85:
        kept = re.sub(r'(\d+\.?\d*)', lambda m: str(round(float(m.group())*0.9,1)), kept)
    return kept

print(f'{"Task":20s} {"ROUGE-L":>8} {"Quant-R":>8} {"Reg-F1":>8} {"Composite":>10}')
print('-'*58)
all_sc = []
for ex in TOX_DATASET:
    pred = sim_pred(ex, 0.78)
    sc   = evaluate_tox_response(pred, ex['output'], ex.get('task','qa'))
    sc['task'] = ex.get('task','qa')
    all_sc.append(sc)
    print(f'{sc["task"]:20s} {sc["rouge_L"]:>8.3f} {sc["quant_recall"]:>8.3f} '
          f'{sc["reg_f1"]:>8.3f} {sc["composite"]:>10.3f}')

import pandas as pd
df = pd.DataFrame(all_sc)
print(f'\nMeans: ROUGE-L={df["rouge_L"].mean():.3f} | Quant={df["quant_recall"].mean():.3f} '
      f'| Reg={df["reg_f1"].mean():.3f} | Composite={df["composite"].mean():.3f}')

---
## Section 10 -- Inference, Deployment & Safety Guardrails

In [ ]:
class ToxLLMInference:
    SAFETY_PATTERNS = [
        (r'administer\s+\d+\s*mg/kg\s+to\s+(?:yourself|patient)', 'CLINICAL_DOSE_ADVICE'),
        (r'(?:definitely|certainly)\s+(?:safe|harmless)', 'OVERCONFIDENT_SAFETY'),
        (r'LD50\s*(?:for|of)\s+humans?\s*(?:is|=)\s*\d+', 'HUMAN_LD50_CLAIM'),
    ]
    CONFIDENCE_SIGNALS = [
        r'\b(?:mg/kg|uM|nM|fold|ppm|ppb)\b',
        r'\b(?:ICH|OECD|EPA|FDA|ECHA|EFSA)\b',
        r'\b(?:NOAEL|LOAEL|BMDL|IC50|EC50|LD50)\b',
        r'\b(?:mechanism|pathway|metabolite)\b',
        r'\b(?:p\s*<|statistically)\b',
    ]

    def __init__(self, model_path=None):
        self.model_path = model_path
        self.model = self.tokenizer = None

    def check_safety(self, text):
        return [name for pat, name in self.SAFETY_PATTERNS
                if re.search(pat, text, re.IGNORECASE)]

    def domain_confidence(self, text):
        hits = sum(1 for p in self.CONFIDENCE_SIGNALS
                   if re.search(p, text, re.IGNORECASE))
        return round(min(1.0, hits/len(self.CONFIDENCE_SIGNALS)*2), 3)

    def generate(self, question, context='', max_new_tokens=400):
        user = question + (f'\n\nContext:\n{context}' if context.strip() else '')
        prompt = (f'<|im_start|>system\n{SYSTEM_PROMPT_TOX}\n<|im_end|>\n'
                  f'<|im_start|>user\n{user}<|im_end|>\n'
                  f'<|im_start|>assistant\n')
        if self.model and self.tokenizer:
            inputs = self.tokenizer(prompt, return_tensors='pt').to(DEVICE)
            with torch.no_grad():
                out = self.model.generate(**inputs, max_new_tokens=max_new_tokens,
                                          do_sample=False,
                                          pad_token_id=self.tokenizer.eos_token_id,
                                          repetition_penalty=1.1)
            response = self.tokenizer.decode(out[0][inputs.input_ids.shape[1]:],
                                             skip_special_tokens=True)
        else:
            q_lower = question.lower()
            response = next((ex['output'][:500] for ex in TOX_DATASET
                            if any(w in q_lower for w in
                                   ex.get('input','').lower().split()[:8] if len(w)>4)),
                            TOX_DATASET[0]['output'][:400])
        safety = self.check_safety(response)
        return {'response': response, 'safety_flags': safety,
                'confidence': self.domain_confidence(response),
                'needs_review': bool(safety) or self.domain_confidence(response) < 0.3}

toxllm = ToxLLMInference()
QUESTIONS = [
    'What is the ICH M7 two-method framework for genotoxic impurity assessment?',
    'Compound with hERG IC50=5uM and free Cmax=2uM -- safety margin and next steps?',
    'Explain the difference between NOAEL and BMD in regulatory risk assessment.',
]

print('ToxLLM Inference Demo'); print('='*65)
for q in QUESTIONS:
    r = toxllm.generate(q)
    print(f'\nQ: {q[:70]}')
    print(f'A: {r["response"][:200]}...')
    print(f'   conf={r["confidence"]} | safety={r["safety_flags"] or "none"} | review={r["needs_review"]}')

In [ ]:
# Deployment options and cheatsheet
lines = [
    'DATASET',
    '  Format:       Alpaca | ChatML | Llama-3',
    '  Min examples: 500-1000 (specialisation), 2000-10000 (production)',
    '  Sources:      REACH dossiers, NTP reports, FDA labels, ICH guidelines',
    '  QC:           length filter > hallucination flag > quality score > dedup',
    '',
    'QLORA (T4 / Colab Free)',
    '  Model:        mistralai/Mistral-7B-Instruct-v0.3 (Apache 2.0)',
    '  Quant:        4-bit NF4, compute bf16 (float16 for T4)',
    '  LoRA rank:    r=32 general | r=64 large domain shift',
    '  Alpha:        64 (scale=2.0)',
    '  Modules:      q,k,v,o,gate,up,down projections',
    '  Batch:        2 per device x 8 grad accum = 16',
    '  LR SFT:       2e-4 | LR DPO: 5e-6',
    '  Epochs SFT:   3    | Epochs DPO: 1',
    '  LR schedule:  cosine with 5% warmup',
    '  CRITICAL:     DataCollatorForCompletionOnlyLM (response-only loss)',
    '',
    'EVALUATION',
    '  ROUGE-L:      lexical overlap (baseline)',
    '  Quant recall: doses/IC50s/fold-changes preserved?',
    '  Reg F1:       correct ICH/OECD citations?',
    '  Composite:    0.30xROUGE + 0.25xQuant + 0.25xReg + 0.20xField',
    '  Benchmark:    zero-shot ~0.28 -> SFT ~0.62 -> DPO ~0.68 ROUGE-L',
    '',
    'DEPLOYMENT',
    '  vLLM:         python -m vllm.entrypoints.openai.api_server --model ./toxllm_merged',
    '  Ollama:       ollama create toxllm -f Modelfile && ollama run toxllm',
    '  HF pipe:      pipeline("text-generation", "./toxllm_merged")',
    '  FastAPI:      wrap generate() in POST endpoint',
    '',
    'TROUBLESHOOTING',
    '  Loss stays high:   wrong prompt template or missing response mask',
    '  Repetition:        add repetition_penalty=1.1 at inference',
    '  Forgets base:      reduce epochs to 2, lower lr to 1e-4',
    '  Hallucinations:    add DPO pairs, increase training data',
    '  Bad JSON:          add more extraction examples + schema in prompt',
]
print('\n'.join(lines))

---
## Dashboard: Fine-Tuning Results Visualisation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Simulated evaluation results (replace with actual trainer.evaluate() outputs)
np.random.seed(42)
stages = ['Zero-shot', 'SFT', 'SFT+DPO']
rouge   = [0.28, 0.62, 0.68]
quant   = [0.41, 0.78, 0.83]
reg_f1  = [0.35, 0.71, 0.76]
composite=[0.34, 0.67, 0.72]

fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

# Panel 1: Performance by stage
ax1 = fig.add_subplot(gs[0, 0])
x = np.arange(len(stages)); w = 0.2
ax1.bar(x-1.5*w, rouge,    w, label='ROUGE-L',    color='#1565C0', alpha=0.85)
ax1.bar(x-0.5*w, quant,    w, label='Quant Recall',color='#27AE60', alpha=0.85)
ax1.bar(x+0.5*w, reg_f1,   w, label='Reg F1',     color='#E74C3C', alpha=0.85)
ax1.bar(x+1.5*w, composite, w, label='Composite',  color='#8E44AD', alpha=0.85)
ax1.set_xticks(x); ax1.set_xticklabels(stages, fontsize=10)
ax1.set_ylim([0, 1.05]); ax1.set_ylabel('Score')
ax1.set_title('Performance by Fine-Tuning Stage', fontweight='bold')
ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3, axis='y')

# Panel 2: Task-level composite scores
ax2 = fig.add_subplot(gs[0, 1])
tasks  = ['qa', 'extraction', 'noael_reasoning', 'ich_compliance', 'iata', 'dose_response', 'classification']
scores = [0.71, 0.65, 0.69, 0.73, 0.67, 0.66, 0.74]
cols   = ['#1565C0' if s>=0.7 else '#E67E22' if s>=0.6 else '#E74C3C' for s in scores]
ax2.barh(tasks, scores, color=cols, alpha=0.85)
ax2.axvline(0.7, color='k', lw=1.2, linestyle='--', alpha=0.5, label='Target 0.70')
ax2.set_xlim([0, 1]); ax2.set_xlabel('Composite Score')
ax2.set_title('SFT+DPO: Scores by Task', fontweight='bold')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3, axis='x')

# Panel 3: DPO preference margin
ax3 = fig.add_subplot(gs[0, 2])
dpo_steps = np.arange(0, 50, 2)
margin    = 0.7 * (1 - np.exp(-dpo_steps/15)) + np.random.normal(0, 0.03, len(dpo_steps))
margin    = np.clip(margin, 0, 1)
ax3.plot(dpo_steps, margin, '#E74C3C', lw=2.2)
ax3.axhline(0.3, color='k', lw=1, linestyle='--', alpha=0.5, label='Stable margin')
ax3.set_xlabel('DPO Steps'); ax3.set_ylabel('Implicit Reward Margin')
ax3.set_title('DPO Training: Preference Margin', fontweight='bold')
ax3.legend(fontsize=9); ax3.grid(True, alpha=0.3)

# Panel 4: Token distribution
ax4 = fig.add_subplot(gs[1, 0])
tok_dist = [int(len(format_chatml(ex).split())*1.3) for ex in TOX_DATASET]
ax4.hist(tok_dist, bins=10, color='#1565C0', alpha=0.8, edgecolor='white')
ax4.axvline(2048, color='#E74C3C', lw=2, linestyle='--', label='2048 limit')
ax4.set_xlabel('Tokens'); ax4.set_ylabel('Examples')
ax4.set_title('Training Data Token Distribution', fontweight='bold')
ax4.legend(fontsize=9); ax4.grid(True, alpha=0.3)

# Panel 5: QC score distribution
ax5 = fig.add_subplot(gs[1, 1])
qc_scores = [0.9, 1.0, 0.85, 0.95, 1.0, 0.9, 0.95, 0.85, 1.0, 0.9,
             0.95, 0.9, 1.0, 0.95]
ax5.bar(range(len(qc_scores)), qc_scores, color='#27AE60', alpha=0.8)
ax5.axhline(0.6, color='#E74C3C', lw=2, linestyle='--', label='Pass threshold 0.6')
ax5.set_xlabel('Example index'); ax5.set_ylabel('QC Score')
ax5.set_title('QC Scores per Training Example', fontweight='bold')
ax5.legend(fontsize=9); ax5.set_ylim([0, 1.1]); ax5.grid(True, alpha=0.3)

# Panel 6: Comparison table
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis('off')
tdata = [
    ['Metric',       'Zero-shot', 'SFT', 'SFT+DPO'],
    ['ROUGE-L',       '0.28',     '0.62', '0.68'],
    ['Quant recall',  '0.41',     '0.78', '0.83'],
    ['Reg F1',        '0.35',     '0.71', '0.76'],
    ['Composite',     '0.34',     '0.67', '0.72'],
    ['Quant gain',    '--',       '+37%', '+42%'],
]
tbl = ax6.table(cellText=tdata[1:], colLabels=tdata[0],
                cellLoc='center', loc='center', bbox=[0,0,1,1])
tbl.auto_set_font_size(False); tbl.set_fontsize(9)
for j in range(4):
    tbl[0,j].set_facecolor('#1565C0')
    tbl[0,j].set_text_props(color='white', fontweight='bold')
for i in range(1,6):
    for j in range(4):
        tbl[i,j].set_facecolor('#EEF5FF' if i%2==0 else 'white')
ax6.set_title('Performance Summary', fontweight='bold', pad=10)

fig.suptitle('ToxLLM Fine-Tuning -- Complete Results Dashboard', fontsize=14, fontweight='bold')
plt.savefig('toxllm_dashboard.png', dpi=130, bbox_inches='tight')
plt.show()
print('Dashboard saved: toxllm_dashboard.png')